# Advanced 02 — Multi-Agent Delegation Security

A framework handoff changes who reasons next. Only a trusted application boundary may decide what the child is allowed to do. This lab attenuates authority across issuance, execution, replay, revocation, and result admission.

![Secure delegation architecture](architecture.svg)

The model and SDK propose routing. The trusted control plane binds identity, attenuates authority, enforces current policy and budget, validates the child result, and emits a content-free receipt.

## 1. Load the course lab

The notebook imports the reusable course module rather than copying its security logic.

In [ ]:
import runpy
from datetime import datetime, timedelta, timezone
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, replace
import asyncio
ns = runpy.run_path('02_multi_agent_security.py')
ActorContext, ArtifactCandidate, EvaluationCase, OperationRequest, WorkerSession = (ns[name] for name in ('ActorContext','ArtifactCandidate','EvaluationCase','OperationRequest','WorkerSession'))
build_scenario, candidate_for, evaluate_cases = (ns[name] for name in ('build_scenario','candidate_for','evaluate_cases'))
now = datetime(2026,9,20,tzinfo=timezone.utc)
service, registry, authority, parent, worker, delegation_request = build_scenario(now=now)
issuance = service.issue(authority, parent, delegation_request, now=now)
assert issuance.allowed and issuance.envelope and service.verify(issuance.envelope)
envelope = issuance.envelope
session = WorkerSession(envelope, service, registry, 'research-runtime')
{'parent_scopes': sorted(authority.scopes), 'child_scopes': sorted(envelope.scopes), 'resource': sorted(envelope.resources), 'audience': envelope.audience, 'depth': envelope.depth}

## 2. Establish the safe baseline

Observe the trusted inputs and the decision evidence before injecting failures.

In [ ]:
valid_request = OperationRequest('op-valid','attempt-valid','search','case:42','evidence-list')
valid = session.execute(worker, valid_request, candidate_for(envelope, valid_request), now=now)
assert valid.allowed and valid.artifact and valid.artifact.evidence_ids
{'decision': valid.reason, 'artifact_digest': valid.artifact.digest, 'receipt': asdict(valid.receipt)}

## 3. Inject an attack

Change one security-relevant boundary and keep the rest of the fixture stable.

In [ ]:
wider_request = OperationRequest('op-delete','attempt-delete','delete','case:42','evidence-list')
wider = session.execute(worker, wider_request, candidate_for(envelope, wider_request), now=now)
assert not wider.allowed and wider.reason == 'scope'
{'allowed': wider.allowed, 'reason': wider.reason, 'budget_after': wider.receipt.budget_after}

## 4. Attempt a bypass

The assertions below make the security property executable and regression-testable.

In [ ]:
cross_resource = OperationRequest('op-cross','attempt-cross','search','case:99','evidence-list')
cross = session.execute(worker, cross_resource, candidate_for(envelope, cross_resource), now=now)
retry_request = replace(valid_request, attempt_id='attempt-retry')
retry = session.execute(worker, retry_request, candidate_for(envelope, retry_request), now=now)
assert not cross.allowed and cross.reason == 'resource'
assert retry.allowed and retry.replayed and session.consumed == 1
{'cross_resource': cross.reason, 'retry': retry.reason, 'consumed': session.consumed}

## 5. Evaluate observable outcomes

Use explicit denominators or counts. Private model reasoning is neither required nor recorded.

In [ ]:
report = evaluate_cases([EvaluationCase('valid search', True, valid), EvaluationCase('scope escalation', False, wider), EvaluationCase('cross-resource request', False, cross)])
assert report.attack_success_rate == 0 and report.blocked_valid_task_rate == 0
{'population': {'attack_cases': report.attack_attempts, 'valid_cases': report.valid_tasks}, 'attack_success_rate': report.attack_success_rate, 'blocked_valid_task_rate': report.blocked_valid_task_rate}

## 6. Exercise a second failure mode

In [ ]:
registry.revoke(envelope.envelope_id)
revoked_retry = replace(valid_request, attempt_id='attempt-after-revocation')
revoked = session.execute(worker, revoked_retry, candidate_for(envelope, revoked_retry), now=now)
assert not revoked.allowed and revoked.reason == 'revoked-lineage'
{'reason': revoked.reason, 'budget_unchanged': session.consumed}

## 7. Tamper and replay resistance

Integrity protects every envelope field. A stable delegation request is idempotent, while the same request ID with changed authority fails closed.

In [ ]:
service2, _, authority2, parent2, _, request2 = build_scenario(now=now)
first = service2.issue(authority2, parent2, request2, now=now)
same = service2.issue(authority2, parent2, request2, now=now)
changed = service2.issue(authority2, parent2, replace(request2, scopes=frozenset({'read'})), now=now)
tampered = replace(first.envelope, budget=99)
assert same.reason == 'idempotent-reissue' and changed.reason == 'request-replay-mismatch'
assert not service2.verify(tampered)
{'same_request': same.reason, 'changed_request': changed.reason, 'tamper_detected': not service2.verify(tampered)}

## 8. Atomic concurrency budget

Eight workers race for two budget units. Authorization and consumption occur in one critical section, so exactly two operations succeed.

In [ ]:
service3, registry3, authority3, parent3, worker3, request3 = build_scenario(now=now)
issued3 = service3.issue(authority3, parent3, request3, now=now)
session3 = WorkerSession(issued3.envelope, service3, registry3, 'research-runtime')
def concurrent_call(index):
    req = OperationRequest(f'op-race-{index}', f'attempt-race-{index}', 'search', 'case:42', 'evidence-list')
    return session3.execute(worker3, req, candidate_for(issued3.envelope, req, content=f'E-{index}'), now=now)
with ThreadPoolExecutor(max_workers=8) as pool:
    raced = list(pool.map(concurrent_call, range(8)))
assert sum(item.allowed for item in raced) == 2 and session3.consumed == 2
{'allowed': sum(item.allowed for item in raced), 'budget_denials': sum(item.reason == 'budget' for item in raced), 'consumed': session3.consumed}

## 9. Result admission

A child result is still untrusted. Producer, tenant, envelope, operation, type, size, and evidence IDs must match before the application computes its digest.

In [ ]:
service4, registry4, authority4, parent4, worker4, request4 = build_scenario(now=now)
issued4 = service4.issue(authority4, parent4, request4, now=now)
session4 = WorkerSession(issued4.envelope, service4, registry4, 'research-runtime')
req4 = OperationRequest('op-result','attempt-result','search','case:42','evidence-list')
forged = replace(candidate_for(issued4.envelope, req4), producer='other-agent')
rejected = session4.execute(worker4, req4, forged, now=now)
assert not rejected.allowed and rejected.reason == 'result-binding'
{'allowed': rejected.allowed, 'reason': rejected.reason, 'artifact': rejected.artifact}

## 10. OpenAI Agents SDK handoff contract

The pinned SDK supplies routing, typed metadata, and history filtering. The callback invokes the same application policy before transfer; this demonstration makes no model or network call.

In [ ]:
sdk = runpy.run_path('02_multi_agent_security_sdk.py')
sdk_handoff, sdk_context = await sdk['credential_free_demo']()
assert sdk_context.last_decision.allowed
{'tool_name': sdk_handoff.tool_name, 'destination': sdk_handoff.agent_name, 'model_fields': sorted(sdk_handoff.input_json_schema['properties']), 'authorization': sdk_context.last_decision.reason}

## 11. Coordination tax

A safer delegation can still be the wrong architecture. Compare its extra boundaries with a single-agent baseline and report wall-clock latency separately from total work in a real evaluation.

In [ ]:
single_agent = {'agents': 1, 'handoffs': 0, 'policy_decisions': 1, 'result_boundaries': 1}
delegated = {'agents': 2, 'handoffs': 1, 'policy_decisions': 2, 'result_boundaries': 2}
coordination_tax = {key: delegated[key] - single_agent[key] for key in single_agent}
coordination_tax

## 12. Production replacement

Production replacement: authenticated workload identity; audience-bound token exchange; asymmetric signing or protected key service; durable transactional budget, fan-out, idempotency, and revocation state; isolated queue, workspace, memory, and egress; resource-level authorization; lineage-aware restart; privacy-aware distributed traces; effect reconciliation; and tested kill switches. The local HMAC and lock prove invariants inside one process, not distributed consistency or key security.

## 13. Exercises

1. Add a second eligible child and prove siblings cannot multiply the parent budget.
2. Derive a depth-two authority and test valid and invented lineage.
3. Replace the in-memory transaction with durable compare-and-swap.
4. Add a cancellation deadline and prove no new provider call starts after it.
5. Build an A2A, LangGraph, Google ADK, or Microsoft Agent Framework adapter that preserves the same boundary.

## Checkpoint

Explain which trusted component enforces the invariant, what evidence proves the decision, and what residual risk remains.